# Raw Data Exploratory Data Analysis (EDA)

**Dataset:** OrionX Labs Travel Expense (Raw Excel)  
**Purpose:** Understand raw data structure, quality issues, and risks prior to cleaning and feature engineering.


In [1]:
import os
os.getcwd()

'/Users/vivekduggal/Documents/Projects/orionx-travel-expense-kpis/notebooks'

In [2]:
import pandas as pd
file_path = "../data/raw/OrionX_Labs_Travel_Expense_Final_70000.xlsx"

df_raw = pd.read_excel(file_path)
df_raw.shape

(70000, 30)

In [5]:
!ls ../data/raw/

OrionX_Labs_Travel_Expense_Final_70000.xlsx


In [6]:
df_raw.head()
df_raw.info()
df_raw.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 30 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   employee                           70000 non-null  object 
 1   employee ID                        70000 non-null  int64  
 2   active                             70000 non-null  object 
 3   department name                    70000 non-null  object 
 4   job family                         70000 non-null  object 
 5   payment type                       70000 non-null  object 
 6   approval status                    70000 non-null  object 
 7   report name                        70000 non-null  object 
 8   report ID                          70000 non-null  object 
 9   parent expense type                70000 non-null  object 
 10  expense type                       70000 non-null  object 
 11  vendor                             70000 non-null  obj

(70000, 30)

In [7]:
(df_raw == "").sum().sort_values(ascending=False).head(10)

employee                        0
employee ID                     0
expense approved amount(rpt)    0
expense approved amount         0
number of attendees             0
reporting currency              0
reimbursement currency          0
subsidiary name                 0
region                          0
country                         0
dtype: int64

## Initial Schema Observations

- Dataset contains 70,000 records and 30 columns
- No null values reported, suggesting upstream curation or placeholder usage
- All date-related fields are stored as strings and require parsing
- Several categorical and boolean fields are stored as object types
- Column naming is inconsistent and contains spaces and special characters


In [10]:
df_raw.describe(include="all").transpose().head(10)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
employee,70000,53191,Michael Smith,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employee ID,70000.0,NaN,NaN,NaN,551202.5913,259357.790055,100017.0,327054.0,552272.5,775807.75,999982.0
active,70000,2,No,35158,NaN,NaN,NaN,NaN,NaN,NaN,NaN
department name,70000,10,Workplace,7181,NaN,NaN,NaN,NaN,NaN,NaN,NaN
job family,70000,5,Software Engineering,34980,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment type,70000,3,Corporate Credit Card,23638,NaN,NaN,NaN,NaN,NaN,NaN,NaN
approval status,70000,4,Sent Back to Employee,17620,NaN,NaN,NaN,NaN,NaN,NaN,NaN
report name,70000,971,Dinner Expense Report,102,NaN,NaN,NaN,NaN,NaN,NaN,NaN
report ID,70000,70000,43451733342082155551,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
parent expense type,70000,2,Travel,35075,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Categorical & Identifier Observations

- Employee names are not unique and should not be used as identifiers
- Employee ID appears to be the correct grouping key
- Report ID is unique per record and should be treated as a string identifier
- Approval status shows a high proportion of "Sent Back to Employee", indicating meaningful workflow friction
- Active vs inactive employee status requires explicit KPI inclusion rules


In [11]:
(df_raw == "").sum().sort_values(ascending=False).head(10)

employee                        0
employee ID                     0
expense approved amount(rpt)    0
expense approved amount         0
number of attendees             0
reporting currency              0
reimbursement currency          0
subsidiary name                 0
region                          0
country                         0
dtype: int64

In [13]:
df_raw[
    ["expense approved amount", "expense approved amount(rpt)"]
].describe(percentiles=[0.5, 0.95, 0.99])

,expense approved amount,expense approved amount(rpt)
count,70000.000000,70000.000000
mean,1510.824217,1511.036448
std,860.599236,983.068731
min,15.020000,7.780000
50%,1508.795000,1387.475000
95%,2851.961500,3342.032500
99%,2970.720100,3969.025900
max,2999.990000,4497.550000


### Conclusion
- Expense amounts are already bounded; outliers are *not* extreme.
- Approved amounts introduce FX-related variance and deserve separate KPI treatment. 
- Median or trimmed mean may be more stable for approved-amount KPIs. 
- Outlier handling can be minimal or KPI-specific rather than global.
- p95 and p99 values are close to the maximum, indicating a hard policy or system cap.
- No negative or zero values were observed in either field.  

**Implication:** Expense amount KPIs are stable and not dominated by extreme outliers. Approved amount KPIs should account for FX-related variance.

In [14]:
df_raw["transaction date"].value_counts().head()

transaction date
11/19/2024    173
09/23/2024    164
10/14/2024    162
10/10/2024    159
09/16/2024    159
Name: count, dtype: int64

In [18]:
df_raw["reimbursement currency"].value_counts()

reimbursement currency
EUR    14106
SGD     7064
GBP     7045
AUD     7014
INR     6986
CAD     6978
BRL     6968
USD     6935
JPY     6904
Name: count, dtype: int64

In [19]:
df_raw["transaction date"].head(10)

0    10/21/2025
1    06/06/2025
2    02/11/2025
3    09/20/2024
4    10/14/2024
5    02/14/2025
6    11/21/2025
7    02/24/2025
8    08/08/2025
9    07/06/2024
Name: transaction date, dtype: object

In [20]:
df_raw['first submitted date'].head(10)

0    10/31/2025
1    06/10/2025
2    02/12/2025
3    09/21/2024
4    10/16/2024
5    02/27/2025
6    11/26/2025
7    02/26/2025
8    08/08/2025
9    07/10/2024
Name: first submitted date, dtype: object

In [21]:
# Parse dates safely(EDA only)
df_dates = df_raw.copy()
date_cols = [
    "transaction date",
    "first submitted date",
    "manager approval date",
    "accounting approval date"
]

for col in date_cols:
    df_dates[col] = pd.to_datetime(df_dates[col], errors="coerce")


In [22]:
df_dates[date_cols].isna().sum()

transaction date            0
first submitted date        0
manager approval date       0
accounting approval date    0
dtype: int64

In [23]:
# Date range sanity checks
df_dates["transaction date"].min(), df_dates["transaction date"].max()

(Timestamp('2024-07-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [24]:
df_dates["first submitted date"].min(), df_dates["first submitted date"].max()

(Timestamp('2024-07-01 00:00:00'), Timestamp('2026-01-14 00:00:00'))

In [25]:
df_dates["manager approval date"].min(), df_dates["manager approval date"].max()

(Timestamp('2024-07-02 00:00:00'), Timestamp('2026-01-21 00:00:00'))

In [26]:
df_dates["accounting approval date"].min(), df_dates["accounting approval date"].max()

(Timestamp('2024-07-04 00:00:00'), Timestamp('2026-01-25 00:00:00'))

### Logical Ordering Validation

trandsaction date  
     < first submitted date  
          < manager approval date  
              < accounting approval date   

In [32]:
(df_dates["first submitted date"] < df_dates["transaction date"]).sum()

np.int64(0)

In [31]:
(df_dates["manager approval date"] < df_dates["first submitted date"]).sum()

np.int64(0)

In [30]:
(df_dates["accounting approval date"] < df_dates["manager approval date"]).sum()

np.int64(0)

### Cross-check derived fields
- transaction month
- transaction quarter
- transaction year

In [34]:
df_dates["parsed_year"] = df_dates["transaction date"].dt.year
(df_dates["parsed_year"] != df_raw["transaction year"]).sum()

np.int64(0)

### Date Validation Summary

- All date fields were validated for format consistency and logical ordering
- Date parsing was performed using defensive coercion to identify invalid values
- Transaction dates fall within an expected historical range
- Logical ordering between transaction, submission, and approval dates was validated
- Any detected inconsistencies will be handled explicitly during data cleaning


### Date Format Observations

- Transaction dates are consistently formatted as MM/DD/YYYY and cluster within the 2024 calendar year
- Submission dates span both 2024 and 2025, indicating significant delays between expense occurrence and submission
- The presence of future-dated submission records relative to transaction dates reflects late reporting behavior rather than data corruption
